# Models

Examples of pre-trained GINs, autoencoders, and scratch GINs.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from lightgbm import LGBMRegressor
from pathlib import Path
from pprint import pprint
from sklearn.metrics import r2_score
import torch
from tqdm import tqdm
import topological_pretraining as tp
import yaml

In [ ]:
data_dir = "../data"
temp_dir = "./temp"
os.makedirs(temp_dir, exist_ok=True)

## Pre-training a GIN on QMugs to predict ECFPs

### Loading QMugs

In [ ]:
qmugs = tp.data.load_dataset("QMugs", root=data_dir, verbose=True)
qmugs.head()

In [ ]:
tanimoto_subset = qmugs[qmugs["tanimoto_filter_0.5"] == True].index
tanimoto_subset = qmugs.rdkit_mols[tanimoto_subset][:10000].tolist()

### Generating substructure tokenized graphs

In [ ]:
# settings for pretraining target
targets = {
    "ECFP": {
        "transform_kwargs": {
            "fpsize": 2048,
            "radius": 2,
            "chirality": True,
        },
    }
}

In [ ]:
# set kwargs for graph featurization using
# the Morgan algorithm with Sort and Slice
transform_kwargs = dict(
    radius=2,
    max_vocab_size=2048,
    chirality=True,
)

In [ ]:
# make a graph dataset with tokens for morgan substructures
# morgan graph featurizer uses sort and slice to rank and select substructures
# fpsize translates to the number of substructures to include in the vocabulary
featurizer = tp.featurization.MorganGraphFeaturizer(transform_kwargs=transform_kwargs, verbose=True)

# create graphs
dataset = tp.data.datasets.GraphDataset(
    root=temp_dir,
    featurizer=featurizer,
    molecules=tanimoto_subset,
    fit_featurizer=True,
    verbose=True,
    targets=targets,
)
dataloader = tp.data.loader.DataLoader(
    dataset, batch_size=512, shuffle=True
    )

In [ ]:
tanimoto_subset[0]

In [ ]:
print(
f"Example graph: {dataset[0]}\n\
0th atom tokens: {dataset[0].x[0]}\n\
0th bond: {dataset[0].edge_index[:,0]}\n\
0th edge tokens: {dataset[0].edge_attr[0]}\n\
ECFP fingerprint: {dataset[0].ECFP}"
)

### Training a GIN to predict ECFPs

In [ ]:
vocab_size = dataset.featurizer.vocab_size

In [ ]:
# initialising GIN and prediction head for ECFP pretraining task
# node_embedding: (number_of_tokens, embedding_dimension)
# input_dim: (number_of_tokens_per_node) - in this case, 3 tokens per node for substructures at 0, 1, and 2 radius
gin_kwargs = dict(
    input_dim=dataset.num_node_features,
    node_embedding=(vocab_size, 128),
    hidden_dim=128,
    num_layers=2, # number of message passing layers
    share_weights=True, # whether to share weights across message passing layers
    act="hardswish",
    layer_pool_type="last",
    graph_pool_type="mean",
    batch_norm=False,
    dropout=0.1,
    gnn_kwargs={
        "num_layers": 2, # number of MLP layers in each GIN layer
        "train_eps": True,
        "weight_init": "xavier_normal",
        "bias_init": "zeros",
    }
)
gin = tp.nn.GIN(
    **gin_kwargs
)
pred_head = tp.nn.BinaryHead(
    input_dim=128,
    hidden_dim=128,
    output_dim=dataset.targets["ECFP"]["transform_kwargs"]["fpsize"],
    num_layers=2,
    dropout=0.1,
    class_weights=None,
)

model = torch.nn.ModuleDict(
    {
        "main": gin,
        "head": pred_head,
    }
)

In [ ]:
# short training loop for ECFP pretraining task
num_epochs = 20
losses = torch.zeros(num_epochs) # to store loss values
scores = torch.zeros(num_epochs) # to store AUCPR
num_parameters = sum(p.numel() for p in model.parameters())
lr = num_parameters ** (-0.5)
optimizer = torch.optim.Adam(
    model.parameters(), lr=lr,
    weight_decay=1e-5
)
model.train()

for epoch in tqdm(range(num_epochs), desc="Training", total=num_epochs):
    for batch in dataloader:
        optimizer.zero_grad()
        # forward pass through GIN
        out = model["main"](
            x=batch.x, 
            edge_index=batch.edge_index,
            batch=batch.batch
        )
        pooled = out["global_state"] # getting graph pooled representation
        pred = model["head"](pooled) # forward pass through prediction head
        # compute loss and AUCPR score
        y = batch["ECFP"]
        y = y.type(pred.dtype)
        loss = model["head"].loss(y=y, pred=pred,)
        # backpropagation and optimization step
        loss.backward()
        optimizer.step()

    # log final batch loss and score for the epoch
    out = model["main"](
        x=batch.x, 
        edge_index=batch.edge_index,
        batch=batch.batch
    )
    pooled = out["global_state"] # getting graph pooled representation
    pred = model["head"](pooled) # forward pass through prediction head
    y = batch["ECFP"]
    y = y.type(pred.dtype)
    losses[epoch] = model["head"].loss(y=y, pred=pred).item()
    scores[epoch] = model["head"].score(y=y, pred=pred).item()

In [ ]:
# plotting loss and AUCPR curves
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(losses)
ax[0].set_title("Loss")
ax[1].plot(scores)
ax[1].set_title("AUCPR")
plt.show()

In [ ]:
# wrap model as pretrained featurizer
model_dict = {
    "featurizer": dataset.featurizer.to_dict(),
    "main": {
        "state": model["main"].state_dict(),
        "cls": model["main"].__class__.__name__,
        "kwargs": gin_kwargs,
    },
}
model = tp.featurization.pretrained.PreTrainedGNN(
    params=model_dict
)

## AUCPR some of our pre-trained models

### GIN

In [ ]:
# loading a pretrained model for inference
pt_gin = "../pt_models/vocab_size/pt_gin_radius_2_vocab_16384.pt"
pt_gin = tp.featurization.pretrained.PreTrainedGNN(path=pt_gin, device="cpu")
pt_gin.eval()

In [ ]:
# getting frozen feature embeddings for the tanimoto subset
embeddings = pt_gin(tanimoto_subset)

In [ ]:
# getting ECFP predictions for the tanimoto subset
ecfp_pred = pt_gin.heads["ECFP"](torch.tensor(embeddings))

In [ ]:
# getting true ECFP fingerprints for the tanimoto subset
ecfps = tp.data.mol.MorganGenerator(radius=2, fpsize=2048, chirality=True)(tanimoto_subset)

In [ ]:
# computing AUCPR for the ECFP predictions
aucpr = pt_gin.heads["ECFP"].score(y=torch.tensor(ecfps), pred=ecfp_pred)

In [ ]:
print(f"AUCPR for ECFP prediction: {aucpr:.4f}")

### Autoencoder

In [ ]:
# loading a pretrained autoencoder for inference
autoencoder_pth = "../pt_models/autoencoder/autoencoder.pt"
autoencoder = tp.featurization.pretrained.PreTrainedModel(path=autoencoder_pth, device="cpu")
autoencoder.eval()

In [ ]:
# getting ECFP predictions for the tanimoto subset
latent_representations = autoencoder(tanimoto_subset)
latent_representations = torch.tensor(latent_representations, dtype=torch.float32)
outputs = autoencoder.model.decoder(latent_representations)

In [ ]:
# computing AUCPR for the autoencoder predictions
ecfps = autoencoder.featurizer.transform(tanimoto_subset)
ecfps = torch.tensor(ecfps, dtype=torch.float32)
aucpr = autoencoder.model.decoder.score(pred=outputs, y=ecfps)

In [ ]:
print(f"AUCPR for ECFP prediction: {aucpr:.4f}")

## Example benchmarking of PT-GIN vs Autoencoder on Solubility

Note: these example results are without hyperparameter tuning.

In [ ]:
# loading the Solubility dataset
solu = tp.data.load_dataset("Solu", root=data_dir, verbose=True)

In [ ]:
# comparing regression performance of pretrained autoencoder and pretrained GIN features on the Solubility dataset
# using lightgbm as the regression model and R² as the evaluation metric
# no hyperparameter tuning or cross validation, just a quick comparison of the two feature sets with a simple regression model
# splits used here are the hp tuning splits used in the final experiments
autoencoder = tp.featurization.pretrained.PreTrainedModel(path=autoencoder_pth, device="cpu")
autoencoder.eval()
pt_gin = tp.featurization.pretrained.PreTrainedGNN(path="../pt_models/vocab_size/pt_gin_radius_2_vocab_2048.pt", device="cpu", layer_pool_type="concat")
pt_gin.eval()
r2_out = np.zeros((5, 3))
for i, (train, test) in tqdm(enumerate(solu.splits), desc="Evaluating", total=5):
    if i == 5: break
    mols = solu.rdkit_mols
    y_train = solu.y[train]
    y_test = solu.y[test]
    # autoencoder
    feat = autoencoder.featurizer.transform(mols)
    feat = torch.tensor(feat, dtype=torch.float32)
    X = autoencoder.model.encoder(feat).detach().numpy()
    X_train = X[train]
    X_test = X[test]
    
    lgbm = LGBMRegressor(n_jobs=-1, verbose=-1)
    lgbm.fit(X_train, y_train)
    y_pred = lgbm.predict(X_test)
    r2_out[i, 0] = r2_score(y_test, y_pred)

    # pretrained GIN
    feat = pt_gin(mols)
    X_train = feat[train]
    X_test = feat[test]
    lgbm = LGBMRegressor(n_jobs=-1, verbose=-1)
    lgbm.fit(X_train, y_train)
    y_pred = lgbm.predict(X_test)
    r2_out[i, 1] = r2_score(y_test, y_pred)

    # newly pretrained GIN
    feat = model(mols)
    X_train = feat[train]
    X_test = feat[test]
    lgbm = LGBMRegressor(n_jobs=-1, verbose=-1)
    lgbm.fit(X_train, y_train)
    y_pred = lgbm.predict(X_test)
    r2_out[i, 2] = r2_score(y_test, y_pred)

In [ ]:
plt.boxplot(r2_out, labels=["Autoencoder", "PT GIN", "Newly PT GIN on 10k molecules"])
plt.ylabel("R²")
plt.title("Solubility performance")
plt.show()

## Scratch GIN

Short example of training two GINs from scratch on solubility task, a larger model and a smaller model. Note that these results are without hyperparameter tuning. 

In [ ]:
# create molecular graphs dataset
molecules = list(solu.rdkit_mols)

small_vocab_dataset = tp.data.datasets.MolDataset(
    mols=solu.rdkit_mols,
    y=solu.y.values,
    featurizer="MorganGraphFeaturizer",
    featurizer_kwargs={'radius': 1, 'max_vocab_size': 128, 'chirality': True},
    verbose=False,
    fit_transform=False
)

large_vocab_dataset = tp.data.datasets.MolDataset(
    mols=solu.rdkit_mols,
    y=solu.y.values,
    featurizer="MorganGraphFeaturizer",
    featurizer_kwargs={'radius': 1, 'max_vocab_size': 2048, 'chirality': True,},
    verbose=False,
    fit_transform=False
)



In [ ]:
# parameters for training GINs from scratch on the Solubility dataset with small and large parameter counts
large_model = dict(
    node_embedding_dim=128,
    hidden_dim=128,
    gnn_layers=3, # number of message passing layers
    mlp_layers=3, # number of MLP layers in each GIN layer
    act="hardswish",
    layer_pool_type="last",
    graph_pool_type="sum",
    dropout=0.2,
    weight_decay=1e-5,
    head_layers=3,
    head_hidden_dim=128,
)
small_model = dict(
    node_embedding_dim=32,
    hidden_dim=32,
    gnn_layers=3, # number of message passing layers
    mlp_layers=2, # number of MLP layers in each GIN layer
    share_weights=True,
    act="hardswish",
    layer_pool_type="last",
    graph_pool_type="sum",
    dropout=0.2,
    weight_decay=1e-5,
    head_layers=2,
    head_hidden_dim=32,
)

In [ ]:
# training scratch GINs on the Solubility task with small and large parameter counts
# no hyperparameter tuning or cross validation, just a brief comparison of larger and smaller models
scratch_r2_out = np.zeros((5, 2))
losses = {
    "small_model": [],
    "large_model": [],
}
for i, (train, test) in tqdm(enumerate(solu.splits), desc="Evaluating", total=5):
    if i == 5: break
    
    small_vocab_dataset.reset(train, test)
    train_X, train_y = small_vocab_dataset.train
    
    model = tp.models.SklearnGIN(
        input_dim=train_X[0].x.size(1),
        vocab_size=small_vocab_dataset.featurizer.vocab_size,
        batch_size=128,
        epochs=50,
        verbose=True,
        return_loss=True,
        task="regression",
        **small_model
    )
    loss = model.fit(train_X, train_y)
    losses["small_model"].append(loss)

    test_X, test_y = small_vocab_dataset.test
    y_pred = model.predict(test_X)
    scratch_r2_out[i, 0] = r2_score(test_y, y_pred)

    large_vocab_dataset.reset(train, test)
    train_X, train_y = large_vocab_dataset.train
    model = tp.models.SklearnGIN(
        input_dim=train_X[0].x.size(1),
        vocab_size=large_vocab_dataset.featurizer.vocab_size,
        batch_size=128,
        epochs=50,
        verbose=True,
        return_loss=True,
        task="regression",
        **large_model
    )
    loss = model.fit(train_X, train_y)
    losses["large_model"].append(loss)
    test_X, test_y = large_vocab_dataset.test
    y_pred = model.predict(test_X)
    scratch_r2_out[i, 1] = r2_score(test_y, y_pred)

    

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for i in range(5):
    ax[0].plot(losses["small_model"][i], label=f"Split {i}")
    ax[1].plot(losses["large_model"][i], label=f"Split {i}")
    
ax[0].set_title("Large Vocab GIN Loss Curves")
ax[1].set_title("Small Vocab GIN Loss Curves")
plt.show()

In [ ]:
# plotting R² scores for the scratch GINs with small and large vocabularies
plt.boxplot(scratch_r2_out, labels=["Small Scratch GIN", "Large Scratch GIN"])
plt.ylabel("R²")
plt.title("Solubility regression performance")
plt.show()

## Running pre-training and benchmarking

In [ ]:
# command to run the pretraining script for training a GIN with a vocabulary size of 2048 and radius of 2 on the tanimoto subset of QMugs
# requires preprocessing of QMugs (see preprocess command in 01_datasets.ipynb)
""" 
!mkdir -p ../results/pretraining
!python ../main.py \
    --config ../config/pretrain/pt_gin/vocab_size/pt_gin_radius_2_vocab_2048.yaml \
    --base_config ../config/pretrain/pt_gin/base.yaml \
    --data ../data \
    --output ../results/pretraining
"""

In [ ]:
# running benchmarking of ECFP fingerprints with LightGBM on Rat_PPB benchmarking task
# requires preprocessing of benchmarking datasets (see preprocess command in 01_datasets.ipynb)

# outputs an npz file for each task with benchmarking predictions for each split

# the first five splits are used for hyperparameter tuning 
# hp splits are included in the npz file but should not be used 
# for evaluating final performance of the model on the benchmarking tasks

# hyperparameters are saved to 
# {data} / hyperparameters / {name parameter in config} / {task}.yaml
# e.g., ../data/hyperparameters/ecfp_benchmark_example/Rat_PPB.yaml
!mkdir -p ../results/benchmarking
!python ../main.py \
    --config ../config/benchmark/ecfp_benchmark_example.yaml \
    --data ../data \
    --output ../results/benchmarking/

In [ ]:
hp_path = Path("../data/hyperparameters/ecfp_benchmark_example/Rat_PPB.yaml")
if hp_path.exists():
    hp = yaml.safe_load(open(hp_path))
    pprint(f"Hyperparameters for Rat_PPB task:\n{hp}")

In [ ]:
results_path = Path("../results/benchmarking/ecfp_benchmark_example/rat_ppb_preds.npz")
if results_path.exists():
    results = np.load(results_path, allow_pickle=False,)['arr_0']
    dataset = tp.data.load_dataset("Rat_PPB", root="../data", verbose=True)
    r2_results = np.zeros(dataset.num_splits)
    for i, (train, test) in enumerate(dataset.splits):
        y_true = dataset.y[test]
        y_pred = results[i, test]
        r2 = r2_score(y_true, y_pred)
        r2_results[i] = r2
    print(
        "Tuning split R² scores:", r2_results[:5].round(4),
        "\nTest split R² mean:", r2_results[5:].mean().round(4),
        "Test split R² std:", r2_results[5:].std().round(4)
    )

In [ ]:
# example of benchmarking pretrained GIN on Rat_PPB task
# limited to 25 splits with no hyperparameter tuning
# requires preprocessing of benchmarking datasets (see preprocess command in 01_datasets.ipynb)
# requires pretrained GIN model (see pretraining command above)
!mkdir -p ../results/benchmarking
!python ../main.py \
    --config ../config/benchmark/pt_gin_benchmark_example.yaml \
    --data ../data \
    --output ../results/benchmarking \
    --model_path ../pt_models/vocab_size/pt_gin_radius_2_vocab_2048.pt

In [ ]:
results_path = Path("../results/benchmarking/pt_gin_benchmark_example/rat_ppb_preds.npz")
if results_path.exists():
    results = np.load(results_path, allow_pickle=False,)['arr_0']
    dataset = tp.data.load_dataset("Rat_PPB", root="../data", verbose=True)
    r2_results = np.zeros(results.shape[0])
    for i, (train, test) in enumerate(dataset.splits):
        if i > results.shape[0] - 1:
            break
        y_true = dataset.y[test]
        y_pred = results[i, test]
        r2 = r2_score(y_true, y_pred)
        r2_results[i] = r2
    print(
        "\nTest split R² mean:", r2_results[5:].mean().round(4),
        "\nTest split R² std:", r2_results[5:].std().round(4)
    )